# T9 It2: can the judge be made less dependent on the input? (attempts) and how big a shift breaks it

It1 found the judge's wrong answers under shifted signals. Attempts, all trained on the NUDT design sets only, cut by Learn-then-Test on NUDT
(floor 0.9), then the same +-1 SD stress on official + NUDT 707:
- *no coarse/sharp*: drop the coarse-search score and sharpness, which depend on the map and photo contrast (the strongest NUDT/official shift);
- *gaps monotone*: also force P to rise with zgap and ligap (uniqueness of the place in its query);
- *relative only*: only signals relative to the photo's own candidates or the fit (no absolute counts).
Note: official and NUDT 707 have been scored before (T8), so this compares variants; it is not a new blind test.

In [1]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "8")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
RES = ROOT / "tests/Test_Iteration_9/results"
import numpy as np, pandas as pd
import train.checks.candidate_judge as cj
from train.checks.holdout import NUDT_DESIGN, load_t8_table
table = load_t8_table(ROOT, locked=False, blind=True)
table.loc[table.data.str.startswith("official"), "data"] = "official"
EVAL = ["official", "nudt707"]
PC = dict(override=12, min_choice=3, wide_tau=0.95)
VARIANTS = {
 "current": (cj.FEATURES_CTX, {"li","z","hull","cover9"}),
 "no coarse/sharp": ([f for f in cj.FEATURES_CTX if f not in ("coarse","sharp")], {"li","z","hull","cover9"}),
 "no coarse/sharp, gaps monotone": ([f for f in cj.FEATURES_CTX if f not in ("coarse","sharp")], {"li","z","hull","cover9","zgap","ligap"}),
 "relative only": (["ligap","zgap","z","hull","cover9","ratio","angle","shift_px","scale_dev","resid","tfrac","hc_ratio"], {"z","hull","cover9","zgap","ligap"}),
}
pd.set_option("display.width", 250)

In [2]:
rows = []
for name, (feats, inc) in VARIANTS.items():
    cj.INCREASING = inc
    p, model = cj.judge_probability(table, NUDT_DESIGN, feats, "gbm")
    # cut: LTT on NUDT design, floor 0.9
    for var, kw in [("pc", {}), ("jetson", dict(jetson=True, m=5))]:
        tau = max(cj.ltt_tau(cj.scan_tau(table[table.data.isin(NUDT_DESIGN)], p[table.data.isin(NUDT_DESIGN).to_numpy()], cj.TAUS, **PC, **kw), NUDT_DESIGN, 0.005), 0.9)
        s = cj.summary(cj.decide(table, p, tau, **PC, **kw))
        # stress: li +1 SD and zgap +-1 SD
        off = table.data.isin(EVAL).to_numpy(); usable = (table.inliers >= 3).to_numpy() & off
        sd = table[table.data.isin(NUDT_DESIGN) & (table.inliers >= 3)][feats].std()
        stress = {}
        for f in feats:
            for sign in (-1, 1):
                X = table.loc[usable, feats].to_numpy(float).copy(); X[:, feats.index(f)] += sign * sd[f]
                p2 = p.copy(); p2[usable] = model.predict_proba(X)[:, 1]
                s2 = cj.summary(cj.decide(table[off], p2[off], tau, **PC, **kw))
                stress[(f, sign)] = int(s2.wrong.sum())
        bad = {k: v for k, v in stress.items() if v > 0}
        rows.append(dict(variant=name, var=var, tau=tau, **{f"s_{d}": s.loc[d, "score"] for d in EVAL + NUDT_DESIGN}, **{f"w_{d}": int(s.loc[d, "wrong"]) for d in EVAL + NUDT_DESIGN},
                         stress_bad=len(bad), stress_worst=max(bad.values()) if bad else 0, stress_list=bad))
o = pd.DataFrame(rows); o.drop(columns="stress_list").to_csv(RES / "it2_hardening.csv", index=False)
print(o.drop(columns="stress_list").round(2).to_string(index=False))
for _, r in o.iterrows(): print(r.variant, r["var"], r.stress_list)

cj.INCREASING = {'li', 'z', 'hull', 'cover9'}

                       variant    var  tau  s_official  s_nudt707  s_nudt101  s_nudt202  s_nudt404  s_nudt505  s_nudt606  w_official  w_nudt707  w_nudt101  w_nudt202  w_nudt404  w_nudt505  w_nudt606  stress_bad  stress_worst
                       current     pc 0.90       53.89      55.44      55.52      55.86      56.52      56.76      56.40           0          0          0          1          0          0          1           8             3
                       current jetson 0.92       52.69      54.84      54.52      54.36      57.12      55.56      55.02           0          0          0          1          0          0          1           6             2
               no coarse/sharp     pc 0.90       54.05      55.44      55.32      55.86      56.52      56.76      56.40           0          0          0          1          0          0          1           9             3
               no coarse/sharp jetson 0.90       53.00      54.84      54.72      54.36      57.30  

## How big is a +1 SD shift, and which cut survives a uniform rise in matches?
The shift that hurts every variant is a **uniform rise of inliers** on every candidate (right and wrong alike), as a more permissive matcher or a
sharper camera would give. Below: the size of 1 SD in inliers, and the wrong answers of the current judge when every candidate's inliers are
multiplied by a factor, at several cuts.

In [3]:
sd_li = table[table.data.isin(NUDT_DESIGN) & (table.inliers >= 3)].li.std()
print("1 SD of log(1 + inliers) = %.2f -> inliers x %.1f" % (sd_li, np.exp(sd_li)))
p0, model = cj.judge_probability(table, NUDT_DESIGN, cj.FEATURES_CTX, "gbm")
off = table.data.isin(EVAL).to_numpy(); sub = table[off].reset_index(drop=True)
rows = []
for factor in [1.0, 1.25, 1.5, 2.0, 3.0]:
    t2 = sub.copy(); t2["inliers"] = np.round(sub.inliers * factor); t2["li"] = np.log1p(t2.inliers)
    u = (t2.inliers >= 3).to_numpy(); p = np.zeros(len(t2)); p[u] = model.predict_proba(t2.loc[u, cj.FEATURES_CTX].to_numpy(float))[:, 1]
    for tau in [0.9, 0.95, 0.98, 0.99]:
        for var, kw in [("pc", {}), ("jetson", dict(jetson=True, m=5))]:
            s = cj.summary(cj.decide(t2, p, tau, **PC, **kw))
            rows.append(dict(factor=factor, tau=tau, var=var, **{f"wrong_{d}": int(s.loc[d, "wrong"]) for d in EVAL}, **{f"score_{d}": s.loc[d, "score"] for d in EVAL}))
INFL = pd.DataFrame(rows); INFL.to_csv(RES / "it2_inlier_inflation.csv", index=False)
INFL.pivot_table(index=["var", "factor"], columns="tau", values=["wrong_official", "wrong_nudt707"]).astype(int)

1 SD of log(1 + inliers) = 1.09 -> inliers x 3.0


wrong_nudt707                wrong_official               
tau                    0.90 0.95 0.98 0.99           0.90 0.95 0.98 0.99
var    factor                                                           
jetson 1.00               0    0    0    0              0    0    0    0
       1.25               2    2    2    2              0    0    0    0
       1.50               2    2    2    2              0    0    0    0
       2.00               6    6    6    6              1    1    1    1
       3.00              14   14   14   14              1    1    1    1
pc     1.00               0    0    0    0              0    0    0    0
       1.25               3    3    2    2              0    0    0    0
       1.50               5    5    4    4              0    0    0    0
       2.00              11   11   11   11              1    2    2    2
       3.00              20   20   20   20              1    1    1    1

In [4]:
INFL.pivot_table(index=["var", "factor"], columns="tau", values=["score_official", "score_nudt707"]).round(2)

score_nudt707                      score_official                     
tau                    0.90   0.95   0.98   0.99           0.90   0.95   0.98   0.99
var    factor                                                                       
jetson 1.00           54.84  54.84  54.84  54.84          52.84  52.43  50.63  49.58
       1.25           54.24  54.24  54.04  54.04          53.32  52.84  52.05  50.63
       1.50           54.24  54.24  54.04  54.04          53.64  53.32  52.69  52.53
       2.00           52.64  52.64  52.64  52.44          55.31  54.36  53.64  53.64
       3.00           49.10  49.10  49.10  49.10          56.11  56.11  56.04  55.88
pc     1.00           55.44  55.44  55.44  55.44          53.89  53.79  51.80  50.76
       1.25           54.64  54.64  54.84  54.84          54.21  54.05  53.54  51.96
       1.50           53.84  53.84  54.24  54.24          55.16  54.84  54.21  53.86
       2.00           51.44  51.44  51.44  51.44          55.95  55.09  54.52  54.21
       3.00           47.24  47.24  47.24  47.24          56.99  56.99  56.83  56.77

## The same inflation for today's plain bars, and for the judge without its 12-inlier override

In [5]:
rows = []
for factor in [1.0, 1.25, 1.5, 2.0]:
    t2 = sub.copy(); t2["inliers"] = np.round(sub.inliers * factor); t2["li"] = np.log1p(t2.inliers)
    u = (t2.inliers >= 3).to_numpy(); p = np.zeros(len(t2)); p[u] = model.predict_proba(t2.loc[u, cj.FEATURES_CTX].to_numpy(float))[:, 1]
    runs = {("pc", "bar 8 / 11"): cj.baseline(t2, 8, 11, min_choice=3), ("jetson", "Jetson today"): cj.baseline(t2, 8, 8, jetson=True, m=5),
            ("pc", "judge 0.9, no override"): cj.decide(t2, p, 0.9, override=None, min_choice=3, wide_tau=0.95),
            ("pc", "judge 0.98, no override"): cj.decide(t2, p, 0.98, override=None, min_choice=3, wide_tau=0.98),
            ("jetson", "judge 0.925, no override"): cj.decide(t2, p, 0.925, override=None, min_choice=3, wide_tau=0.95, jetson=True, m=5)}
    for (var, name), r in runs.items():
        s = cj.summary(r)
        rows.append(dict(factor=factor, var=var, rule=name, **{f"wrong_{d}": int(s.loc[d, "wrong"]) for d in EVAL}, **{f"score_{d}": s.loc[d, "score"] for d in EVAL}))
INFL2 = pd.DataFrame(rows); INFL2.to_csv(RES / "it2_inlier_inflation_bars.csv", index=False)
INFL2.pivot_table(index=["var", "rule"], columns="factor", values=["wrong_nudt707", "wrong_official"]).astype(int)

wrong_nudt707                wrong_official               
factor                                   1.00 1.25 1.50 2.00           1.00 1.25 1.50 2.00
var    rule                                                                               
jetson Jetson today                         2    6    7   14              0    1    1    1
       judge 0.925, no override             0    1    1    1              0    0    0    0
pc     bar 8 / 11                           3    5    9   18              0    1    1    1
       judge 0.9, no override               0    2    2    3              0    0    0    0
       judge 0.98, no override              0    0    0    0              0    0    0    0